# W02 — ML Task Framing

## 1) My lane as an ML task

My chosen lane is **content refresh / declining content**.

I frame this as a **classification** problem. The unit of analysis is one content page, and the goal is to classify whether a page is showing a declining trend that should be prioritized for human review and possible content refresh.

The model output would support a content team's decision about which pages to investigate and refresh first.

This model would be used as a prioritization aid rather than an automatic decision-maker. A human should review the recommendation before taking action.

## 2) Target or proxy

The target is whether a content page is showing a declining trend.

In the starter data, the `trend_direction` column provides an observable proxy for this outcome. I will use the value `down` to represent a declining page.

The target can therefore be represented as:

- `1` = declining (`trend_direction == "down"`)
- `0` = not declining

This target is a proxy for the decision of whether a page deserves attention for a possible content refresh. It does not prove that refreshing the page will improve performance.

## 3) Success metric

I will use **F1-score** as the primary evaluation metric.

Both types of mistakes matter in this problem. A false negative could cause a genuinely declining page to be missed, while a false positive could cause the content team to spend time reviewing a page that does not need attention.

F1-score balances precision and recall, making it appropriate for a prioritization problem where both missed opportunities and unnecessary reviews have a cost.

The model should ultimately be evaluated not only by its metric, but also by whether its recommendations are useful for the content team's workflow.

## 4) The unit of analysis, as a real dataframe

The unit of analysis is **one content page**. Each row in the dataset represents one content page, with page-level attributes such as its performance and trend information.

I will inspect the actual dataframe to confirm the available page-level features and show examples of individual observations.

In [6]:
import pandas as pd

df = pd.read_csv(
    r"D:\flyrank\flyrank-ml-main\flyrank-ml-main\data\raw\content_refresh_anonymized.csv"
)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 4) The unit of analysis, as a real dataframe

The unit of analysis is **one content page**. Each row in the dataframe represents one content page identified by `content_id`.

The dataset contains 30,000 content-page observations and 44 columns describing characteristics and performance measures such as search volume, competition, content type, word count, CTR, average position, engagement rate, AI traffic percentage, impression tier, position tier, and trend information.

The `trend_direction` column provides an observed indication of whether a page is declining, stable, or improving.

The dataframe below shows actual examples of these page-level observations.

In [7]:
print("Number of pages:", len(df))
print("Number of features:", len(df.columns))

df[[
    "content_id",
    "client_id",
    "content_type",
    "search_volume",
    "word_count",
    "ctr",
    "avg_position",
    "trend_direction",
    "trend_pct"
]].head(10)

Number of pages: 30000
Number of features: 44


,content_id,client_id,content_type,search_volume,word_count,ctr,avg_position,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,10.0,3221.0,0.76,10.6,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,90.0,2481.0,0.05,20.3,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.0,3515.0,0.09,36.5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,10.0,NaN,0.49,6.2,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.0,2803.0,0.13,44.0,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,keyword article,720.0,3080.0,0.03,8.5,down,-38.9
6,content_9a34b442b552,client_8722616204,keyword article,0.0,3059.0,0.00,7.0,down,-92.3
7,content_a63219c6e95a,client_19581e27de,keyword article,590.0,NaN,0.06,21.2,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,keyword article,0.0,3807.0,0.09,46.0,down,-58.8
9,content_c27558df2b0c,client_19581e27de,keyword article,0.0,NaN,0.16,4.9,down,-29.2


In [8]:
print("Trend direction counts:")
print(df["trend_direction"].value_counts())

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [9]:
print("Declining pages:", (df["trend_direction"] == "down").sum())
print("Declining percentage:", round(
    (df["trend_direction"] == "down").mean() * 100, 2
), "%")

Declining pages: 16262
Declining percentage: 54.21 %


In [10]:
df["trend_direction"].value_counts()

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

## 5) Why ML beats a fixed rule here

A simple fixed rule could identify declining pages by checking whether `trend_direction == "down"`. However, that rule only describes the observed trend; it does not help determine which declining pages should be prioritized when a content team has limited time.

An ML approach could use multiple page-level features, such as search volume, competition, content characteristics, CTR, average position, engagement rate, and other available signals, to learn patterns associated with declining pages.

The model could therefore provide a prioritization signal for human review rather than relying on a single fixed condition.

ML is useful here because the decision may depend on combinations of several features rather than one rule. However, the model would not automatically decide that a page should be refreshed, and it would not prove that refreshing a page will improve its performance.

## 6) Self-check

- [x] I named the ML task type: classification.
- [x] I identified the target/proxy: declining content (`trend_direction == "down"`).
- [x] I selected F1-score as the primary success metric.
- [x] I defined the unit of analysis: one content page.
- [x] I showed the real starter dataset as a dataframe.
- [x] I connected the output to a real action: prioritize pages for human review and possible refresh.
- [x] I explained why ML could add value beyond a simple fixed rule.
- [x] I used real numbers from the starter data.
- [x] I avoided claiming that the model proves a content refresh will improve performance.
- [x] I treat the model as a decision-support tool rather than an automatic decision-maker.s